# Extractive Summarizer: SBERT Fine-Tuning + K-Means + Post-Filtering
### Notebook Thử nghiệm & Đánh giá Hoàn chỉnh (Sẵn sàng chạy trên Colab)

Notebook này thực hiện:
1. **Fine-Tuning SBERT** sử dụng tập cặp câu Oracle & `CosineSimilarityLoss` (Giải quyết góp ý Giữa kỳ của Giảng viên).
2. **Khung Đánh giá Kép (Dual-Evaluation Framework)**: So sánh Lead-3, TextRank, Pretrained SBERT và Fine-Tuned SBERT trên các chỉ số Nội tại (Silhouette, Diversity) và Ngoại tại (ROUGE-1, ROUGE-2, ROUGE-L).
3. **Demo Tóm tắt Trực tiếp** với tính năng tự động nhận diện ngôn ngữ và hiển thị kết quả.

In [ ]:
# 1. Cài đặt Môi trường & Thư viện (Hỗ trợ Google Colab)
import os
import sys

# Tự động clone repository nếu chạy trên Google Colab
if 'colab' in str(get_ipython()):
    if not os.path.exists('Extractive-Summarizer'):
        !git clone https://github.com/kttt294/Extractive-Summarizer.git
        %cd Extractive-Summarizer
    !pip install -q -r requirements.txt

# Thêm thư mục gốc vào đường dẫn Python
sys.path.append(os.getcwd())

## 2. Fine-Tuning Mô hình SBERT (Giải quyết Góp ý Giữa kỳ)
Fine-tune SBERT trên tập cặp câu Oracle được trích xuất từ dataset bài báo tin tức.

In [ ]:
from src.train import train_finetune_sbert

# Chạy Fine-tuning SBERT cho Tiếng Anh / Tiếng Việt trong 2 epochs
model_path = train_finetune_sbert(lang='en', epochs=2, batch_size=16, sample_data_count=100)

## 3. Khung Đánh giá Kép (Chỉ số Nội tại & Ngoại tại)
So sánh kết quả giữa Lead-3, TextRank, SBERT Gốc và SBERT Fine-Tuned trên tập thử nghiệm.

In [ ]:
from src.evaluate import evaluate_framework

# Chạy đánh giá trên 30 mẫu thử nghiệm
evaluate_framework(lang='en', sample_count=30)

## 4. Demo Tóm tắt Văn bản Trực tiếp
Chạy thử nghiệm pipeline tóm tắt trích xuất trên một bài báo thực tế.

In [ ]:
from src.preprocess import resolve_language, preprocess_text
from src.evaluate import run_sbert_pipeline

sample_article = '''
Trí tuệ nhân tạo đang thay đổi các ngành công nghiệp trên toàn cầu với tốc độ chưa từng có.
Những tiến bộ gần đây trong xử lý ngôn ngữ tự nhiên cho phép máy tính hiểu và tóm tắt các tài liệu phức tạp chỉ trong vài giây.
Kỹ thuật tóm tắt trích xuất chọn trực tiếp các câu chứa nhiều thông tin nhất từ văn bản gốc.
Bằng cách tận dụng vector nhúng Sentence-BERT, máy tính có thể nắm bắt mối quan hệ ngữ nghĩa sâu sắc giữa các câu.
Thuật toán K-Means giúp gom nhóm các khái niệm tương đồng, đảm bảo bản tóm tắt cuối cùng bao quát nhiều góc độ khác nhau.
Kỹ thuật lọc sau (Post-filtering) tiếp tục loại bỏ các thông tin trùng lặp, tạo ra bản tóm tắt ngắn gọn và giàu giá trị cho người đọc.
Mô hình kết hợp này đại diện cho một phương pháp tiếp cận mạnh mẽ cho các hệ thống tóm tắt văn bản tự động hiện đại.
'''

lang = resolve_language(sample_article, user_choice='auto')
summary_text, summary_sents, sil_score, div_score = run_sbert_pipeline(sample_article, lang=lang, use_finetuned=True)

print(f"Ngôn ngữ tự động nhận diện: {lang.upper()}")
print(f"Chỉ số Nội tại (Intrinsic): Silhouette Score = {sil_score:.4f} | Diversity Score = {div_score:.4f}")
print("\n--- KẾT QUẢ TÓM TẮT TRÍCH XUẤT ---")
for i, sent in enumerate(summary_sents, 1):
    print(f"{i}. {sent}")
